<a href="https://colab.research.google.com/github/isg-data/insurance_eda/blob/main/EDA_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
dir()

[1] "sample_data"

In [ ]:
# ============================================
# EDA: Medical Cost Personal Dataset
# ============================================

# Pakete laden (in Colab ggf. vorher installieren)
# install.packages(c("tidyverse"))
library(tidyverse)

# ---------------------------------------------
# 1. Daten laden
# ---------------------------------------------
# Passe den Pfad an dein Repo an, z.B. "data/insurance.csv"
df <- read.csv("insurance.csv", stringsAsFactors = FALSE)

# Erster Überblick
str(df)
head(df)
summary(df)

# Fehlende Werte prüfen
colSums(is.na(df))

# Datentypen anpassen (kategoriale Variablen als Faktoren)
df <- df %>%
  mutate(
    sex = as.factor(sex),
    smoker = as.factor(smoker),
    region = as.factor(region)
  )

# ---------------------------------------------
# 2. Balkendiagramm: smoker vs. charges
#    (Durchschnittliche Kosten je Rauchstatus)
# ---------------------------------------------
smoker_summary <- df %>%
  group_by(smoker) %>%
  summarise(
    mean_charges = mean(charges),
    median_charges = median(charges),
    n = n()
  )

print(smoker_summary)

ggplot(smoker_summary, aes(x = smoker, y = mean_charges, fill = smoker)) +
  geom_col(width = 0.6) +
  geom_text(aes(label = round(mean_charges, 0)), vjust = -0.5) +
  labs(
    title = "Durchschnittliche Versicherungskosten: Raucher vs. Nichtraucher",
    x = "Raucherstatus",
    y = "Durchschnittliche Kosten (charges)"
  ) +
  scale_fill_manual(values = c("no" = "#4C9F70", "yes" = "#D6604D")) +
  theme_minimal() +
  theme(legend.position = "none")

# ---------------------------------------------
# 3. Liniendiagramm: bmi vs. charges
#    (nach BMI sortiert, da Liniendiagramme eine
#     sinnvolle Reihenfolge auf der x-Achse brauchen)
# ---------------------------------------------

# Variante A: geglättete Trendlinie (empfohlen für Rohdaten)
ggplot(df, aes(x = bmi, y = charges)) +
  geom_point(alpha = 0.3, color = "grey50") +
  geom_smooth(method = "loess", color = "steelblue", se = TRUE) +
  labs(
    title = "Zusammenhang zwischen BMI und Versicherungskosten",
    x = "BMI",
    y = "Kosten (charges)"
  ) +
  theme_minimal()

# Variante B: "echte" Linie über nach BMI gruppierte Mittelwerte
# (bessere Wahl, falls explizit ein klassisches Liniendiagramm gefordert ist)
bmi_binned <- df %>%
  mutate(bmi_bin = round(bmi)) %>%
  group_by(bmi_bin) %>%
  summarise(mean_charges = mean(charges))

ggplot(bmi_binned, aes(x = bmi_bin, y = mean_charges)) +
  geom_line(color = "steelblue", linewidth = 1) +
  geom_point(color = "steelblue") +
  labs(
    title = "Durchschnittliche Kosten je BMI-Wert",
    x = "BMI",
    y = "Durchschnittliche Kosten (charges)"
  ) +
  theme_minimal()